# OpenAI Fine-Tuning Setup

**dc_dalin - Week 6 Contribution**

This notebook demonstrates fine-tuning a closed-source model (GPT-4o-mini) using OpenAI's Fine-tuning API.

## Overview

We'll fine-tune GPT-4o-mini to become a specialized trivia expert, then compare its performance against the base model.

In [ ]:
import os
import json
import time
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

## Step 1: Validate Training Data

First, let's check our training and validation datasets.

In [ ]:
# Validate training data format
def validate_jsonl(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()
        print(f"\n{file_path}:")
        print(f"Total examples: {len(lines)}")
        
        # Show first example
        first = json.loads(lines[0])
        print(f"\nFirst example:")
        print(json.dumps(first, indent=2))
        
        return len(lines)

train_count = validate_jsonl('fine_tune_train.jsonl')
val_count = validate_jsonl('fine_tune_validation.jsonl')

print(f"\n✅ Data validation complete!")
print(f"Training examples: {train_count}")
print(f"Validation examples: {val_count}")

## Step 2: Upload Training Files to OpenAI

We need to upload our training and validation files to OpenAI before creating the fine-tuning job.

In [ ]:
# Upload training file
print("Uploading training file...")
with open('fine_tune_train.jsonl', 'rb') as f:
    training_file = client.files.create(
        file=f,
        purpose='fine-tune'
    )

print(f"✅ Training file uploaded: {training_file.id}")
print(f"   Status: {training_file.status}")
print(f"   Filename: {training_file.filename}")

# Upload validation file
print("\nUploading validation file...")
with open('fine_tune_validation.jsonl', 'rb') as f:
    validation_file = client.files.create(
        file=f,
        purpose='fine-tune'
    )

print(f"✅ Validation file uploaded: {validation_file.id}")
print(f"   Status: {validation_file.status}")
print(f"   Filename: {validation_file.filename}")

## Step 3: Create Fine-Tuning Job

Now we'll create the fine-tuning job using GPT-4o-mini as the base model.

In [ ]:
# Create fine-tuning job
print("Creating fine-tuning job...")

fine_tune_job = client.fine_tuning.jobs.create(
    training_file=training_file.id,
    validation_file=validation_file.id,
    model="gpt-4o-mini-2024-07-18",
    suffix="trivia-expert-dc-dalin",
    hyperparameters={
        "n_epochs": 3,  # Number of training epochs
    }
)

print(f"\n✅ Fine-tuning job created!")
print(f"   Job ID: {fine_tune_job.id}")
print(f"   Status: {fine_tune_job.status}")
print(f"   Model: {fine_tune_job.model}")
print(f"   Created at: {datetime.fromtimestamp(fine_tune_job.created_at)}")

# Save job ID for later reference
job_id = fine_tune_job.id
print(f"\n💾 Save this Job ID: {job_id}")

## Step 4: Monitor Fine-Tuning Progress

Fine-tuning can take several minutes to hours depending on the dataset size. Let's monitor the progress.

In [ ]:
# Monitor fine-tuning progress
def monitor_fine_tuning(job_id, check_interval=60):
    """
    Monitor fine-tuning job status.
    
    Args:
        job_id: Fine-tuning job ID
        check_interval: Seconds between status checks
    """
    print(f"Monitoring job {job_id}...\n")
    
    while True:
        job = client.fine_tuning.jobs.retrieve(job_id)
        status = job.status
        
        timestamp = datetime.now().strftime("%H:%M:%S")
        print(f"[{timestamp}] Status: {status}")
        
        if status == "succeeded":
            print(f"\n✅ Fine-tuning completed successfully!")
            print(f"   Fine-tuned model: {job.fine_tuned_model}")
            
            # Save the model ID
            with open('fine_tuned_model_id.txt', 'w') as f:
                f.write(job.fine_tuned_model)
            
            print(f"\n💾 Model ID saved to 'fine_tuned_model_id.txt'")
            return job.fine_tuned_model
        
        elif status == "failed":
            print(f"\n❌ Fine-tuning failed!")
            if job.error:
                print(f"   Error: {job.error}")
            return None
        
        elif status == "cancelled":
            print(f"\n⚠️  Fine-tuning was cancelled")
            return None
        
        # Show progress if available
        if hasattr(job, 'trained_tokens') and job.trained_tokens:
            print(f"   Trained tokens: {job.trained_tokens:,}")
        
        time.sleep(check_interval)

# Start monitoring
fine_tuned_model_id = monitor_fine_tuning(job_id, check_interval=30)

## Step 5: Retrieve Job Details and Metrics

Once the fine-tuning is complete, let's examine the training metrics.

In [ ]:
# Get final job details
job = client.fine_tuning.jobs.retrieve(job_id)

print("\n📊 Fine-Tuning Job Summary")
print("=" * 50)
print(f"Job ID: {job.id}")
print(f"Status: {job.status}")
print(f"Base Model: {job.model}")
print(f"Fine-tuned Model: {job.fine_tuned_model}")
print(f"Training File: {job.training_file}")
print(f"Validation File: {job.validation_file}")
print(f"Created: {datetime.fromtimestamp(job.created_at)}")
print(f"Finished: {datetime.fromtimestamp(job.finished_at) if job.finished_at else 'N/A'}")

if job.trained_tokens:
    print(f"\nTraining Tokens: {job.trained_tokens:,}")

print("\n" + "=" * 50)

## Step 6: List All Fine-Tuned Models

You can also view all your fine-tuned models.

In [ ]:
# List all fine-tuning jobs
jobs = client.fine_tuning.jobs.list(limit=5)

print("\n📋 Recent Fine-Tuning Jobs")
print("=" * 80)

for job in jobs.data:
    print(f"\nJob ID: {job.id}")
    print(f"  Status: {job.status}")
    print(f"  Base Model: {job.model}")
    if job.fine_tuned_model:
        print(f"  Fine-tuned Model: {job.fine_tuned_model}")
    print(f"  Created: {datetime.fromtimestamp(job.created_at)}")

## Step 7: Test the Fine-Tuned Model

Let's do a quick test to compare the base model vs fine-tuned model.

In [ ]:
# Quick comparison test
test_question = "What is the capital of France?"

print("\n🧪 Quick Test: Base vs Fine-Tuned Model")
print("=" * 50)
print(f"Question: {test_question}\n")

# Base model
base_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a trivia expert that provides accurate, concise answers."},
        {"role": "user", "content": test_question}
    ],
    max_tokens=50,
    temperature=0
)

print(f"Base Model (gpt-4o-mini):")
print(f"  Answer: {base_response.choices[0].message.content}")

# Fine-tuned model
if fine_tuned_model_id:
    ft_response = client.chat.completions.create(
        model=fine_tuned_model_id,
        messages=[
            {"role": "system", "content": "You are a trivia expert that provides accurate, concise answers."},
            {"role": "user", "content": test_question}
        ],
        max_tokens=50,
        temperature=0
    )
    
    print(f"\nFine-Tuned Model ({fine_tuned_model_id}):")
    print(f"  Answer: {ft_response.choices[0].message.content}")
else:
    print("\n⚠️  Fine-tuned model not available yet")

## Next Steps

Now that you have a fine-tuned model:

1. ✅ Copy the fine-tuned model ID from `fine_tuned_model_id.txt`
2. ✅ Use it in `trivia_challenge_with_finetuning.ipynb` to compare performance
3. ✅ Analyze the differences between base and fine-tuned models

## Cost Estimation

Fine-tuning costs for GPT-4o-mini (as of 2024):
- Training: $3.00 / 1M tokens
- Usage: Input $0.30 / 1M tokens, Output $1.20 / 1M tokens

This small dataset (~50 examples) should cost less than $1 to fine-tune.

---

**dc_dalin - Week 6 Contribution** - Demonstrating closed-source model fine-tuning with OpenAI